In [ ]:
import os
import pandas as pd
import scanpy as sc

# Initialize directory for cohort-internal variations
zikv_group_dir = os.path.join(base_dir, "ZIKV_group_DEG")
os.makedirs(zikv_group_dir, exist_ok=True)

# Define pairing parameters within the infection group to pinpoint direct viral pressure variants
intra_zikv_pairs = [
    ('ZIKV_Infected RG', 'ZIKV_Active-state RG', 'Intra_RG_Infect_vs_Active'),
    ('ZIKV_Infected IPCs', 'ZIKV_IPCs', 'Intra_IPCs_Infect_vs_Normal'),
    ('ZIKV_Infected Neurons', 'ZIKV_Newborn Neurons', 'Intra_Neurons_Infect_vs_Newborn')
]

core_label_genes_intra = ['HNRNPD', 'HNRNPA1', 'ISG15', 'IFI27', 'DDIT3', 'BAX', 'EOMES', 'PAX6']

print("Calculating sample-internal specific DEG vectors (Infected vs Bystander)...")
excel_path_intra = os.path.join(zikv_group_dir, "ZIKV_Intra_Sample_DEG_Analysis.xlsx")

with pd.ExcelWriter(excel_path_intra) as writer:
    for test_grp, ref_grp, key_name in intra_zikv_pairs:
        if test_grp in adata.obs['group_for_deg'].values and ref_grp in adata.obs['group_for_deg'].values:
            print(f"Processing intra-group comparison: {test_grp} vs {ref_grp}...")
            
            sc.tl.rank_genes_groups(
                adata, groupby='group_for_deg', groups=[test_grp], reference=ref_grp, 
                method='wilcoxon', key_added=key_name
            )
            
            df_res = sc.get.rank_genes_groups_df(adata, group=test_grp, key=key_name)
            sheet = key_name.replace('Intra_', '')[:31]
            df_res.to_excel(writer, sheet_name=sheet, index=False)
            
            save_pdf_intra = os.path.join(zikv_group_dir, f"Volcano_{key_name}_NatureStyle.pdf")
            internal_nature_volcano(
                df_res, 
                f"ZIKV Group Intra-Comparison:\n{test_grp.split('_')[1]} vs Bystander", 
                save_pdf_intra, 
                core_label_genes_intra
            )
            print(f"Intra-volcano plot saved: {save_pdf_intra}")
        else:
            print(f"Skipping intra-vector: Target subpopulation missing: {test_grp} or {ref_grp}")